# Sistemas Multiagente

Um agente trabalha em um contexto só: o prompt, as ferramentas e tudo o que ele leu ficam na mesma conversa. Um **sistema multiagente** reparte a tarefa entre agentes, cada um com o próprio contexto. Este notebook checa três afirmações sobre Natal com um verificador individual e com uma equipe de pesquisadores e editor, e compara as duas versões em Python.

In [1]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q langchain langchain-openai

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [2]:
from typing import Literal

import pandas as pd
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

In [3]:
model = init_chat_model("openai:gpt-4.1-mini", temperature=0.0, use_responses_api=True)

web_search = {"type": "web_search"}

In [4]:
pd.set_option("display.max_colwidth", None)


def user_message(text):
    """Monta a entrada do agente com uma mensagem do usuário."""
    return {"messages": [{"role": "user", "content": text}]}


def usage(results):
    """Soma chamadas ao modelo, buscas na web e tokens de entrada de uma ou mais execuções."""
    calls, searches, tokens = 0, 0, 0
    for result in results:
        for message in result["messages"]:
            if message.type == "ai":
                calls += 1
                tokens += message.usage_metadata["input_tokens"]
                searches += sum(block["type"] == "server_tool_call" for block in message.content_blocks)
    return {"chamadas": calls, "buscas": searches, "tokens": tokens}

## Agente único

A busca na web da OpenAI é uma ferramenta do provedor: ela roda no servidor, dentro da chamada ao modelo. O verificador individual recebe as três afirmações na mesma conversa e devolve um `Report`, o esquema passado em `response_format`, com um veredito para cada afirmação.

In [5]:
CLAIMS = [
    "O Instituto Metrópole Digital da UFRN foi criado em 2011.",
    "A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.",
    "No Censo de 2022, Natal tinha cerca de 751 mil habitantes.",
]

In [6]:
class Check(BaseModel):
    claim: str = Field(description="A afirmação checada")
    fact: str = Field(description="O dado correto, segundo a fonte")
    source: str = Field(description="Só a URL da fonte")
    verdict: Literal["VERDADEIRO", "FALSO", "NÃO VERIFICADO"]


class Report(BaseModel):
    checks: list[Check]


def show_report(report):
    """Exibe os vereditos de um relatório como tabela."""
    return pd.DataFrame([check.model_dump() for check in report.checks])

In [7]:
VERIFIER_PROMPT = """# Papel
Você é um verificador de fatos.

# Tarefa
Pesquise na web cada afirmação recebida e dê o veredito pelo que a busca trouxe.
Sem resultado de busca sobre a afirmação, o veredito é NÃO VERIFICADO."""

verifier_agent = create_agent(
    model,
    tools=[web_search],
    system_prompt=VERIFIER_PROMPT,
    response_format=Report,
)

In [8]:
%%time
verifier_result = verifier_agent.invoke(user_message("\n".join(CLAIMS)))

CPU times: user 212 ms, sys: 5.64 ms, total: 217 ms
Wall time: 3.2 s


In [9]:
for message in verifier_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

O Instituto Metrópole Digital da UFRN foi criado em 2011.
A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.
No Censo de 2022, Natal tinha cerca de 751 mil habitantes.
================================== Ai Message ==================================

[{'id': 'ws_06c1ba6426c2d490006afb39904e7c87d286266afea165b14b', 'action': {'query': 'Instituto Metrópole Digital UFRN criação', 'type': 'search', 'queries': ['Instituto Metrópole Digital UFRN criação']}, 'status': 'completed', 'type': 'web_search_call'}, {'type': 'text', 'text': '{"checks":[{"claim":"O Instituto Metrópole Digital da UFRN foi criado em 2011.","fact":"O Instituto Metrópole Digital da UFRN foi criado em 18 de abril de 2011 como unidade suplementar da universidade, embora seu projeto exista desde 2009.","source":"([portal.imd.ufrn.br](https://portal.imd.ufrn.br/portal/institucional/historico?utm_source=openai))","verdict":"VERDADEIRO"},{"c

In [10]:
usage([verifier_result])

{'chamadas': 1, 'buscas': 1, 'tokens': 8174}

In [11]:
verifier_report = verifier_result["structured_response"]
show_report(verifier_report)

,claim,fact,source,verdict
0,O Instituto Metrópole Digital da UFRN foi criado em 2011.,"O Instituto Metrópole Digital da UFRN foi criado em 18 de abril de 2011 como unidade suplementar da universidade, embora seu projeto exista desde 2009.",([portal.imd.ufrn.br](https://portal.imd.ufrn.br/portal/institucional/historico?utm_source=openai)),VERDADEIRO
1,"A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.","A Ponte Newton Navarro, em Natal, foi inaugurada em 21 de novembro de 2007.",([es.wikipedia.org](https://es.wikipedia.org/wiki/Puente_Newton_Navarro?utm_source=openai)),FALSO
2,"No Censo de 2022, Natal tinha cerca de 751 mil habitantes.",Não foi possível encontrar informações específicas sobre a população de Natal no Censo de 2022 nos resultados disponíveis.,N/A,NÃO VERIFICADO


A execução do verificador tem uma única mensagem de IA, ou seja, uma chamada ao modelo, e a busca aparece dentro dela como `web_search_call`, sem mensagem de ferramenta. Essa chamada respondeu às três afirmações com menos buscas que afirmações, e a afirmação sem dado no resultado da busca ficou como NÃO VERIFICADO.

## Paralelização

As três afirmações não dependem uma da outra. Na **paralelização**, agentes independentes rodam ao mesmo tempo: o método `batch` roda um pesquisador por afirmação, em threads paralelas. Cada pesquisador investiga uma única afirmação e responde com notas em texto.

In [12]:
RESEARCHER_PROMPT = """# Papel
Você é um pesquisador.

# Tarefa
Pesquise na web a afirmação recebida.
Responda com o dado correto, o veredito (VERDADEIRO ou FALSO) e a URL da fonte."""

researcher_agent = create_agent(
    model,
    tools=[web_search],
    system_prompt=RESEARCHER_PROMPT,
)

In [14]:
%%time
requests = [user_message(claim) for claim in CLAIMS]
research_results = researcher_agent.batch(requests)

CPU times: user 34.1 ms, sys: 8.68 ms, total: 42.8 ms
Wall time: 2.78 s


In [15]:
for result in research_results:
    for message in result["messages"]:
        message.pretty_print()

================================ Human Message =================================

O Instituto Metrópole Digital da UFRN foi criado em 2011.
================================== Ai Message ==================================

[{'id': 'ws_02c75df9416adf01006afb3993018c87d2ad33842b408f03b7', 'action': {'query': 'Instituto Metrópole Digital UFRN data de criação', 'type': 'search', 'queries': ['Instituto Metrópole Digital UFRN data de criação']}, 'status': 'completed', 'type': 'web_search_call'}, {'type': 'text', 'text': 'A afirmação é VERDADEIRA. O Instituto Metrópole Digital (IMD) da Universidade Federal do Rio Grande do Norte (UFRN) foi criado em 18 de abril de 2011 como unidade suplementar da UFRN. No entanto, seu projeto existe desde 2009. Na ocasião de sua inauguração, o Instituto já desenvolvia atividades de ensino no Curso Técnico de Formação de Programadores, contemplando 1.200 vagas voltadas para jovens com idades entre 14 e 18 anos regularmente matriculados no Ensino Médio, sendo 70

In [16]:
pd.DataFrame([usage([result]) for result in research_results], index=CLAIMS)

,chamadas,buscas,tokens
O Instituto Metrópole Digital da UFRN foi criado em 2011.,1,1,8174
"A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.",1,1,8174
"No Censo de 2022, Natal tinha cerca de 751 mil habitantes.",1,1,8174


Cada pesquisador tem a própria conversa, com uma chamada ao modelo e uma busca sobre a sua afirmação, e as três conversas não trocaram nada entre si. Cada chamada leu tantos tokens quanto a do verificador, porque os resultados da busca entram no contexto do modelo. O `batch` levou perto do tempo de uma chamada só, já que as três rodaram ao mesmo tempo.

### Exercício 1

Rode os pesquisadores em sequência, com um laço de `invoke` sobre as `requests`, em uma célula com `%%time`. Quantas vezes mais lento ficou em relação ao `batch`?

In [ ]:
# Seu código aqui

## Isolamento de contexto

No **isolamento de contexto**, um agente devolve só o resultado, e o que ele leu fica no contexto dele. O editor abaixo não tem acesso à web: recebe apenas as notas dos pesquisadores e as unifica em um parecer, no mesmo esquema `Report` do verificador individual.

In [18]:
EDITOR_PROMPT = """# Papel
Você é um editor de checagem de fatos.

# Tarefa
Unifique as notas dos pesquisadores em um parecer, com um veredito por afirmação.
Use só as informações das notas."""

editor_agent = create_agent(model, system_prompt=EDITOR_PROMPT, response_format=Report)

In [19]:
notes = ""
for claim, result in zip(CLAIMS, research_results):
    notes += f"# Afirmação: {claim}\n{result['messages'][-1].text}\n\n"

In [20]:
%%time
editor_result = editor_agent.invoke(user_message(notes))

CPU times: user 20.9 ms, sys: 1.55 ms, total: 22.4 ms
Wall time: 3.53 s


In [21]:
for message in editor_result["messages"]:
    message.pretty_print()

================================ Human Message =================================

# Afirmação: O Instituto Metrópole Digital da UFRN foi criado em 2011.
A afirmação é VERDADEIRA. O Instituto Metrópole Digital (IMD) da Universidade Federal do Rio Grande do Norte (UFRN) foi criado em 18 de abril de 2011 como unidade suplementar da UFRN. No entanto, seu projeto existe desde 2009. Na ocasião de sua inauguração, o Instituto já desenvolvia atividades de ensino no Curso Técnico de Formação de Programadores, contemplando 1.200 vagas voltadas para jovens com idades entre 14 e 18 anos regularmente matriculados no Ensino Médio, sendo 70% delas destinadas aos alunos do ensino público. ([metropoledigital.ufrn.br](https://metropoledigital.ufrn.br/portal/institucional/historico?utm_source=openai)) 

# Afirmação: A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.
A Ponte Newton Navarro, em Natal, foi inaugurada em 21 de novembro de 2007. ([tribunadonorte.com.br](https://tribunadonorte.com.br/na

In [22]:
editor_report = editor_result["structured_response"]
show_report(editor_report)

,claim,fact,source,verdict
0,O Instituto Metrópole Digital da UFRN foi criado em 2011.,"O Instituto Metrópole Digital (IMD) da UFRN foi criado em 18 de abril de 2011 como unidade suplementar da UFRN, embora seu projeto exista desde 2009.",https://metropoledigital.ufrn.br/portal/institucional/historico?utm_source=openai,VERDADEIRO
1,"A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.","A Ponte Newton Navarro, em Natal, foi inaugurada em 21 de novembro de 2007.",https://tribunadonorte.com.br/natal/inaugurada-nesta-terca-a-ponte-newton-navarro/?utm_source=openai,FALSO
2,"No Censo de 2022, Natal tinha cerca de 751 mil habitantes.","De acordo com o Censo Demográfico de 2022 do IBGE, Natal tinha uma população de 751.300 habitantes.",https://www.ibge.gov.br/cidades-e-estados/rn/natal.html?utm_source=openai,VERDADEIRO


A mensagem que o editor recebeu traz só as notas, sem os resultados das buscas, e a resposta é o parecer em JSON, no mesmo esquema do verificador individual. Cada veredito e cada fonte do parecer vêm de uma das notas.

## Avaliação

A avaliação é código Python, sem agente. O gabarito `EXPECTED` traz o veredito correto de cada afirmação, e a tabela abaixo compara com ele os vereditos do verificador individual e da equipe.

In [23]:
EXPECTED = ["VERDADEIRO", "FALSO", "VERDADEIRO"]

evaluation = pd.DataFrame({
    "afirmação": CLAIMS,
    "gabarito": EXPECTED,
    "verificador": [check.verdict for check in verifier_report.checks],
    "equipe": [check.verdict for check in editor_report.checks],
})
evaluation

,afirmação,gabarito,verificador,equipe
0,O Instituto Metrópole Digital da UFRN foi criado em 2011.,VERDADEIRO,VERDADEIRO,VERDADEIRO
1,"A Ponte Newton Navarro, em Natal, foi inaugurada em 2010.",FALSO,FALSO,FALSO
2,"No Censo de 2022, Natal tinha cerca de 751 mil habitantes.",VERDADEIRO,NÃO VERIFICADO,VERDADEIRO


In [24]:
summary = pd.DataFrame(
    [usage([verifier_result]), usage(research_results + [editor_result])],
    index=["verificador", "equipe"],
)
summary["acertos"] = [
    (evaluation["verificador"] == evaluation["gabarito"]).sum(),
    (evaluation["equipe"] == evaluation["gabarito"]).sum(),
]
summary

,chamadas,buscas,tokens,acertos
verificador,1,1,8174,2
equipe,4,3,25117,3


A equipe acertou mais vereditos que o verificador individual, porque cada afirmação teve uma busca própria. Em troca, fez mais chamadas ao modelo, leu mais tokens e levou mais tempo no total, como mostram as saídas de `%%time`: o editor só começa depois que o último pesquisador termina.

### Exercício 2

Crie outro verificador individual com um prompt que exija uma busca separada para cada afirmação e rode-o sobre as `CLAIMS`. Quantas chamadas ao modelo e quantas buscas ele fez, e quantos vereditos acertou?

In [ ]:
# Seu código aqui